# LTX Video Test

Prueba [Lightricks/LTX-Video](https://huggingface.co/Lightricks/LTX-Video) (2B params).
Carga cada componente individualmente con `device_map="cuda"` para evitar RAM del sistema.

In [ ]:
# ---- 1. INSTALAR ----
import os, torch
!pip install -qU diffusers transformers accelerate sentencepiece
!pip install -q imageio[ffmpeg] pillow psutil

In [ ]:
# ---- 2. IMPORTS ----
import torch, gc, psutil
from diffusers.utils import load_image, export_to_video
from IPython.display import Video
from transformers import T5EncoderModel, T5TokenizerFast
from diffusers import (
    LTXPipeline,
    AutoencoderKLLTXVideo,
    LTXVideoTransformer3DModel,
    FlowMatchEulerDiscreteScheduler,
)

DTYPE = torch.bfloat16
ram = psutil.virtual_memory()
print(f'RAM total: {ram.total/1024**3:.1f} GB, libre: {ram.available/1024**3:.1f} GB')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {vram:.1f} GB')

## Carga componente a componente directo a GPU

Cada submodelo se carga con `device_map="cuda"` para que accelerate envie los pesos directo a VRAM sin pasar por RAM del sistema.

In [ ]:
# ---- 3. CARGAR COMPONENTES ----

gc.collect(); torch.cuda.empty_cache()

MODEL_ID = 'Lightricks/LTX-Video'

print('Cargando VAE...')
vae = AutoencoderKLLTXVideo.from_pretrained(
    MODEL_ID, subfolder='vae',
    torch_dtype=DTYPE, device_map='cuda',
)
print(f'  RAM libre: {psutil.virtual_memory().available/1024**3:.1f} GB')

print('Cargando Transformer...')
transformer = LTXVideoTransformer3DModel.from_pretrained(
    MODEL_ID, subfolder='transformer',
    torch_dtype=DTYPE, device_map='cuda',
)
print(f'  RAM libre: {psutil.virtual_memory().available/1024**3:.1f} GB')

print('Cargando Text Encoder (T5)...')
text_encoder = T5EncoderModel.from_pretrained(
    MODEL_ID, subfolder='text_encoder',
    torch_dtype=DTYPE, device_map='cuda',
)
print(f'  RAM libre: {psutil.virtual_memory().available/1024**3:.1f} GB')

print('Cargando Tokenizer + Scheduler...')
tokenizer = T5TokenizerFast.from_pretrained(MODEL_ID, subfolder='tokenizer')
scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(MODEL_ID, subfolder='scheduler')

print('Armando pipeline...')
pipe = LTXPipeline(
    vae=vae,
    transformer=transformer,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    scheduler=scheduler,
)
pipe.vae.enable_tiling()
pipe.enable_attention_slicing()
print('OK')

## Text-to-Video

In [ ]:
# ---- 4. TEXT-TO-VIDEO ----

with torch.inference_mode():
    output = pipe(
        prompt='A cinematic drone shot of a misty mountain range at sunrise.',
        width=512, height=288,
        num_frames=9,
        num_inference_steps=20,
        guidance_scale=5.0,
    )

export_to_video(output.frames[0], '/content/ltx_t2v.mp4', fps=24)
print('OK - /content/ltx_t2v.mp4')
Video('/content/ltx_t2v.mp4', embed=True, width=512)

## Image-to-Video

In [ ]:
# ---- 5. IMAGE-TO-VIDEO ----

from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from diffusers import LTXConditionPipeline

print('Armando LTXConditionPipeline...')
pipe_c = LTXConditionPipeline(
    vae=vae,
    transformer=transformer,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    scheduler=scheduler,
)
pipe_c.vae.enable_tiling()
pipe_c.enable_attention_slicing()

image = load_image('https://huggingface.co/datasets/huggingface/documentation-images/'
                    'resolve/main/diffusers/guitar-man.png')
cond = LTXVideoCondition(image=image, frame_index=0)

with torch.inference_mode():
    output = pipe_c(
        prompt='A man playing guitar, slow motion.',
        conditions=[cond],
        width=512, height=288,
        num_frames=9,
        num_inference_steps=20,
        guidance_scale=5.0,
    )

export_to_video(output.frames[0], '/content/ltx_i2v.mp4', fps=24)
print('OK - /content/ltx_i2v.mp4')
Video('/content/ltx_i2v.mp4', embed=True, width=512)

In [ ]:
# ---- 6. LIMPIEZA ----
del pipe, pipe_c, vae, transformer, text_encoder, tokenizer, scheduler
gc.collect(); torch.cuda.empty_cache()
print('Done')